# 03 — Agrupamiento K-means

One-hot + edad escalada, TruncatedSVD de 50 componentes (con el porcentaje real de varianza retenida), selección de k ∈ {2,…,10} por el rango promedio de Silhouette, Davies–Bouldin y Calinski–Harabasz (k-means++, n_init = 20, semilla 42) y recomendación Top-5 por frecuencia de medicamentos en cada conglomerado de Entrenamiento.

In [ ]:
!pip -q install scikit-learn pandas numpy

In [ ]:
import io, os, json, random, warnings
import numpy as np, pandas as pd
from google.colab import files
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold, KFold, RandomizedSearchCV, train_test_split
warnings.filterwarnings("ignore")
RS = 42
random.seed(RS); np.random.seed(RS)
SPLIT = 'fila'   # 'fila' = análisis principal del artículo | 'grupo' = sensibilidad agrupada por episodio
FEATURES = ['EDAD', 'SEXO', 'TIPOURGENCIA', 'AFECPRIN', 'AFEC']
TARGET = 'MEDICAMENTOS'
KS = (1, 3, 5)
LOG = {}
print("Sube Tacotalpa2024.csv")
up = files.upload(); CSV = next(iter(up))
OUT = f"salida_{SPLIT}"
os.makedirs(OUT, exist_ok=True)

## Preparación común (idéntica en los cuatro notebooks)
Limpieza, episodio aproximado, descripción de clases y partición 80/20 (semilla 42). Con `SPLIT='fila'` se reproduce la partición del artículo (6,277 / 1,560); con `SPLIT='grupo'`, la partición agrupada por episodio usada como análisis de sensibilidad.

In [ ]:
# ---------------------------------------------------------------- 1. Flujo de datos
df = pd.read_csv(CSV, encoding="latin1")
flow = [("Registros en el archivo integrado (3 hospitales, 2020-2022)", len(df))]
for c in ['AFECPRIN', 'AFEC', 'MEDICAMENTOS', 'DESCRIPCION', 'SEXO', 'TIPOURGENCIA']:
    df[c] = df[c].astype("string").str.strip()
df['EDAD'] = pd.to_numeric(df['EDAD'], errors='coerce')
n0 = len(df)
df = df.dropna(subset=FEATURES + [TARGET]).copy()
flow.append(("Excluidos por valores faltantes en variables esenciales", -(n0 - len(df))))
# Episodio aproximado: filas consecutivas con la misma EDAD, SEXO, TIPOURGENCIA y AFECPRIN (el archivo
# integrado conserva el orden de la unión URGENCIAS-AFECCIONES-MEDICAMENTOS, pero no el ID original).
# AFEC no forma parte de la clave porque la unión genera el producto AFEC x MEDICAMENTO dentro de un episodio.
EP_KEY = ['EDAD', 'SEXO', 'TIPOURGENCIA', 'AFECPRIN']
key = df[EP_KEY].astype(str).agg('|'.join, axis=1)
df['EPISODIO'] = (key != key.shift()).cumsum().astype(int)
ndup = int(df.duplicated(subset=FEATURES + [TARGET, 'DESCRIPCION']).sum())
df = df.drop_duplicates(subset=FEATURES + [TARGET, 'DESCRIPCION']).reset_index(drop=True)
flow.append(("Excluidos por duplicados exactos", -ndup))
# Un medicamento repetido dentro de un mismo episodio (distinta presentación) se cuenta una vez
# para el nivel episodio; a nivel fila se conserva como en el protocolo original.
flow.append(("Conjunto analítico (filas episodio-medicamento)", len(df)))
LOG['n_analitico'] = len(df)
LOG['n_episodios'] = int(df.EPISODIO.nunique())
LOG['n_perfiles_unicos'] = int(df[FEATURES].astype(str).agg('|'.join, axis=1).nunique())
ep_size = df.groupby('EPISODIO')[TARGET].nunique()
LOG['episodios_con_1_med'] = int((ep_size == 1).sum())
LOG['episodios_con_2plus_med'] = int((ep_size >= 2).sum())
LOG['filas_en_episodios_multi'] = int(df.EPISODIO.map(df.groupby('EPISODIO').size()).gt(1).sum())
LOG['episodios_multi_afec'] = int((df.groupby('EPISODIO').AFEC.nunique() > 1).sum())
LOG['max_meds_episodio'] = int(ep_size.max())

# ---------------------------------------------------------------- 2. Descripción de clases
vc = df[TARGET].value_counts()
LOG['n_clases'] = int(len(vc))
LOG['clases_1_obs'] = int((vc == 1).sum())
LOG['clases_lt5'] = int((vc < 5).sum())
LOG['clases_lt10'] = int((vc < 10).sum())
for k in (1, 3, 5, 10, 20):
    LOG[f'prop_top{k}'] = float(vc.head(k).sum() / len(df))
p = vc / vc.sum()
LOG['entropia_bits'] = float(-(p * np.log2(p)).sum())
LOG['perplejidad'] = float(2 ** LOG['entropia_bits'])
vc.rename_axis('Medicamento').reset_index(name='Frecuencia').assign(
    Porcentaje=lambda d: 100 * d.Frecuencia / len(df),
    Acumulado=lambda d: d.Porcentaje.cumsum()).to_csv(f"{OUT}/distribucion_medicamentos.csv", index=False)

# ---------------------------------------------------------------- 3. Partición
singleton = vc[vc < 2].index
ep_single = set(df.loc[df[TARGET].isin(singleton), 'EPISODIO'])
if SPLIT == 'grupo':
    pool = df[~df.EPISODIO.isin(ep_single)]
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RS)
    tr_idx, te_idx = next(sgkf.split(pool, pool[TARGET], groups=pool.EPISODIO))
    test_df = pool.iloc[te_idx].copy()
    train_df = pd.concat([pool.iloc[tr_idx], df[df.EPISODIO.isin(ep_single)]]).copy()
else:  # protocolo original: estratificado por fila, sin agrupar
    df_single = df[df[TARGET].isin(singleton)]
    df_split = df[~df[TARGET].isin(singleton)]
    tr, test_df = train_test_split(df_split, test_size=.2, random_state=RS, stratify=df_split[TARGET])
    train_df = pd.concat([tr, df_single]).copy()
train_df = train_df.reset_index(drop=True); test_df = test_df.reset_index(drop=True)
LOG['split'] = SPLIT
LOG['n_train'] = len(train_df); LOG['n_test'] = len(test_df)
LOG['ep_train'] = int(train_df.EPISODIO.nunique()); LOG['ep_test'] = int(test_df.EPISODIO.nunique())
LOG['ep_compartidos'] = len(set(train_df.EPISODIO) & set(test_df.EPISODIO))
LOG['filas_singleton_train'] = int(df[TARGET].isin(singleton).sum())
LOG['filas_en_ep_singleton_train'] = int(df.EPISODIO.isin(ep_single).sum())
LOG['clases_test_no_vistas'] = int((~test_df[TARGET].isin(set(train_df[TARGET]))).sum())
flow.append(("Entrenamiento (filas)", len(train_df)))
flow.append(("Prueba interna reservada (filas)", len(test_df)))
pd.DataFrame(flow, columns=['Etapa', 'Registros']).to_csv(f"{OUT}/flujo_datos.csv", index=False)

# Representación a nivel episodio para la prueba: un perfil y el conjunto de medicamentos.
test_ep = test_df.groupby('EPISODIO').agg(**{c: (c, 'first') for c in FEATURES},
                                           MEDS=(TARGET, lambda s: set(s))).reset_index()
LOG['test_episodios'] = len(test_ep)

In [ ]:
# ---------------------------------------------------------------- utilidades de métricas
def row_hits(y, recs):
    """Matriz por caso: rango (1..5) del medicamento verdadero o 0 si no aparece."""
    return np.array([(r.index(t) + 1) if t in r else 0 for t, r in zip(y, recs)])


def set_ranks(sets, recs):
    """Nivel episodio: rango del primer medicamento relevante del episodio."""
    out = []
    for s, r in zip(sets, recs):
        pos = [i + 1 for i, m in enumerate(r) if m in s]
        out.append(pos[0] if pos else 0)
    return np.array(out)


def summarize(ranks, recs, sets=None):
    d = {f'Hit@{k}': float(np.mean((ranks > 0) & (ranks <= k))) for k in KS}
    d['MRR'] = float(np.mean(np.where(ranks > 0, 1 / np.maximum(ranks, 1), 0)))
    d['Cobertura'] = float(np.mean([len(r) > 0 for r in recs]))
    if sets is not None:  # recall@5 a nivel episodio
        d['Recall@5'] = float(np.mean([len(set(r[:5]) & s) / len(s) for s, r in zip(sets, recs)]))
    return d

In [ ]:
RECS = {}  # (recomendaciones por fila, recomendaciones por episodio)
test_ep_ids = test_ep.EPISODIO

## Representación, selección de k y recomendación

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
cat_cols = ['SEXO', 'TIPOURGENCIA', 'AFECPRIN', 'AFEC']

pre_km = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
                            ('age', StandardScaler(with_mean=False), ['EDAD'])])
Xtr_s = pre_km.fit_transform(train_df[FEATURES])
LOG['km_dim_onehot'] = int(Xtr_s.shape[1])
svd = TruncatedSVD(n_components=min(50, Xtr_s.shape[1] - 1), random_state=RS)
Xtr = svd.fit_transform(Xtr_s)
LOG['svd_var_50'] = float(svd.explained_variance_ratio_.sum())
# componentes necesarios para 80/90/95 % (SVD amplio sólo para el reporte)
svd_big = TruncatedSVD(n_components=min(600, Xtr_s.shape[1] - 1), random_state=RS).fit(Xtr_s)
cum = np.cumsum(svd_big.explained_variance_ratio_)
for th in (0.8, 0.9, 0.95):
    LOG[f'svd_comp_{int(th*100)}'] = int(np.searchsorted(cum, th) + 1) if cum[-1] >= th else None
pd.DataFrame({'componentes': np.arange(1, len(cum) + 1), 'varianza_acumulada': cum}).to_csv(
    f"{OUT}/svd_varianza_acumulada.csv", index=False)
rows = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, init='k-means++', n_init=20, random_state=RS)
    lab = km.fit_predict(Xtr)
    rows.append({'k': k, 'Silhouette': silhouette_score(Xtr, lab, sample_size=min(3000, len(Xtr)), random_state=RS),
                 'DaviesBouldin': davies_bouldin_score(Xtr, lab), 'CalinskiHarabasz': calinski_harabasz_score(Xtr, lab),
                 'Inercia': km.inertia_})
kt = pd.DataFrame(rows)
kt['rank_promedio'] = (kt.Silhouette.rank(ascending=False) + kt.DaviesBouldin.rank() + kt.CalinskiHarabasz.rank(ascending=False)) / 3
kt.to_csv(f"{OUT}/kmeans_seleccion_k.csv", index=False)
BEST_K = int(kt.sort_values(['rank_promedio', 'k']).iloc[0].k)
LOG['km_k'] = BEST_K
LOG['km_scores'] = kt[kt.k == BEST_K].iloc[0].drop('k').to_dict()
kmeans = KMeans(n_clusters=BEST_K, init='k-means++', n_init=20, random_state=RS).fit(Xtr)
tr_cl = kmeans.labels_
LOG['km_cluster_sizes'] = pd.Series(tr_cl).value_counts().sort_index().tolist()
cl_rank = {c: train_df.loc[tr_cl == c, TARGET].value_counts().index.tolist() for c in range(BEST_K)}


def km_top5(X):
    cl = kmeans.predict(svd.transform(pre_km.transform(X[FEATURES])))
    return [cl_rank[int(c)][:5] for c in cl]


RECS['K-means'] = (km_top5(test_df), km_top5(test_ep))
display(kt.round(4))
print('k =', BEST_K, '| varianza SVD (50 comp.):', round(100*LOG['svd_var_50'], 1), '% | 90 %:', LOG['svd_comp_90'], 'comp. | 95 %:', LOG['svd_comp_95'], 'comp. | tamaños:', LOG['km_cluster_sizes'])

## Desempeño en Prueba

In [ ]:
rk = row_hits(test_df[TARGET].tolist(), RECS['K-means'][0])
display(pd.DataFrame([summarize(rk, RECS['K-means'][0])]).round(4))

## Exportación

In [ ]:
pred = pd.DataFrame({'EPISODIO': test_df.EPISODIO, 'Real': test_df[TARGET],
                     'Top5': [' | '.join(r) for r in RECS['K-means'][0]]})
pred.to_csv(f"{OUT}/kmeans_predicciones_test.csv", index=False)
with open(f"{OUT}/kmeans_resumen.json", "w", encoding="utf-8") as f:
    json.dump(LOG, f, ensure_ascii=False, indent=2, default=float)
print(json.dumps(LOG, ensure_ascii=False, indent=1, default=float))
files.download(f"{OUT}/kmeans_predicciones_test.csv"); files.download(f"{OUT}/kmeans_resumen.json")
files.download(f"{OUT}/kmeans_seleccion_k.csv")